<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/14-agent-steps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vectara Agent Steps: Deterministic Plan Execution

This notebook shows how to use **agent steps** to drive an LLM agent through a fixed sequence of phases — making the agent's flow deterministic where it matters, while still letting the model do the work inside each phase.

A *step* is a phase of an agent session with its own system prompt, allowed tools, allowed skills, and structured-output schema. The `steps` map plus `first_step_name` lets you wire phases together with conditional `next_steps` transitions. Crucially, **session history is preserved across step transitions** — only the system prompt changes — so a later phase can read everything earlier phases produced.

You'll learn how to:
1. Build a sequential **classify → extract → flag_issues** pipeline (a plan-then-execute pattern)
2. Trace `step_transition` and `structured_output` events to see the pipeline in motion
3. Add a **conditional gate** that bypasses the pipeline for inputs that aren't actually contracts
4. Use **`reentry_step`** so follow-up user turns enter a Q&A phase instead of restarting the pipeline
5. Decide between steps, sub-agents, skills, and a single comprehensive prompt

> **How this differs from notebook 10.** Notebook 10 covers the **classifier-router fan-out** pattern: one classifier branches to one of N terminal handlers. This notebook covers **sequential pipelines** (each phase chains into the next), **conditional gating** (skip a path based on prior output), and **`reentry_step`** (multi-turn flows). The two notebooks are complementary — read both for the full step-orchestration picture.

## Why steps?

A single-step agent compresses every phase of a workflow into one system prompt: *"first decide if this is a contract, then if it is extract these fields, then flag issues, but if it isn't a contract say so politely…"*. Two problems:

1. **The LLM has to remember to do everything.** Long prompts are easy to skim; subtle phases get dropped.
2. **You have no control over flow.** If the model decides to skip extraction and jump straight to flagging, there's nothing to stop it.

Steps fix both. Each phase has:
- **Its own system prompt** (focused on one job — classify, extract, flag).
- **Its own allowed tools / allowed skills** (extract probably shouldn't be calling external APIs).
- **Its own structured-output schema** (the model emits JSON the next phase can route on).
- **Its own `next_steps` rules** (the *system*, not the model, decides what runs next).

The result is a pipeline that's deterministic where it matters (which phase runs, with what tools) while letting the LLM do the actual work inside each phase.

### Step transitions, briefly

After each step's LLM turn completes, Vectara evaluates that step's `next_steps` array in order:
- Each entry has a `step_name` and an optional `condition` written as a UserFn expression.
- The first matching condition wins. An entry without a condition acts as a catch-all.
- If no condition matches, the agent stops in the current step and the turn ends.

Conditions reach into a small context object via `get('$.<path>')`. The fields that matter for routing:
- `$.output.text` — the step's text output (when `output_parser` is `default`)
- `$.output.<field>` — structured-output fields (when `output_parser` is `structured`)
- `$.session.metadata.<key>` and `$.agent.metadata.<key>` — your own metadata
- `$.tools.<tool_config_name>.outputs.latest.<field>` — latest tool output

A single user turn can chain up to **500** transitions before the platform raises `step_transition_limit_exceeded`. Real workflows rarely exceed a dozen — that limit is a safety net for accidental loops, not a budget.

## Getting Started

All you need for this notebook is a `VECTARA_API_KEY`. The notebook is self-contained — it does not depend on the corpora, agents, or tools created in earlier notebooks.

## Setup

In [1]:
import os
import json
from datetime import datetime

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

## Step 1: Build a Three-Phase Pipeline

We'll build a **Contract Triage** agent that processes inbound documents through three sequential phases:

1. **`classify`** — what kind of document is this? Output is JSON: `{doc_type, confidence, reasoning}`. No tools.
2. **`extract`** — pull the key fields a paralegal would need. Output is JSON: `{parties, effective_date, term, governing_law, termination_clause, notes}`. No tools.
3. **`flag_issues`** — given the type and extracted fields (already in session history), produce a risk-flag report. Free-form Markdown. No tools. Terminal step (no `next_steps`).

Each phase has a *focused* system prompt — a model in `classify` is told to do nothing but classify, even though it can technically see the full document. That's the value of phase-structured prompting: the model isn't trying to do five jobs at once.

Transitions:

- `classify` → `extract` (catch-all, no condition)
- `extract` → `flag_issues` (catch-all, no condition)
- `flag_issues` is terminal

We'll add a conditional gate to the `classify` → `extract` transition in Step 3.

In [3]:
CONTRACT_AGENT_NAME = "Contract Triage"

# Schemas for the two structured-output steps. Strict mode requires:
# - additionalProperties: false on all object types
# - all properties listed in `required`
# - no minLength/maxLength/pattern/min/max/items keywords
classify_schema = {
    "name": "contract_classification",
    "description": "Classifies an inbound document by contract type.",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "doc_type": {
                "type": "string",
                "enum": ["nda", "msa", "sow", "vendor_agreement", "other"],
                "description": "The contract type. Use 'other' if the input is not a contract at all."
            },
            "confidence": {
                "type": "number",
                "description": "Calibrated confidence in [0.0, 1.0]."
            },
            "reasoning": {
                "type": "string",
                "description": "One short sentence explaining the classification."
            },
        },
        "required": ["doc_type", "confidence", "reasoning"],
        "additionalProperties": False,
    },
}

extract_schema = {
    "name": "contract_fields",
    "description": "Extracts key fields from a contract.",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "parties": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Named parties to the agreement, in document order.",
            },
            "effective_date": {
                "type": "string",
                "description": "Effective date as written, or '<unknown>' if not stated.",
            },
            "term": {
                "type": "string",
                "description": "Term of the agreement (e.g., '2 years', 'perpetual'), or '<unknown>'.",
            },
            "governing_law": {
                "type": "string",
                "description": "Governing law (jurisdiction) as written, or '<unknown>'.",
            },
            "termination_clause": {
                "type": "string",
                "description": "Brief summary of the termination clause, or '<unknown>'.",
            },
            "notes": {
                "type": "string",
                "description": "Any one-line note worth flagging during extraction (often empty).",
            },
        },
        "required": ["parties", "effective_date", "term", "governing_law", "termination_clause", "notes"],
        "additionalProperties": False,
    },
}

contract_agent_config = {
    "name": CONTRACT_AGENT_NAME,
    "description": "Three-phase contract pipeline: classify -> extract -> flag_issues.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "classify",
    "steps": {
        "classify": {
            "instructions": [{"type": "inline", "name": "classify_instructions", "template": (
                "You are a contract classifier. Look at the inbound text the user just provided and classify "
                "it into one of: 'nda' (any kind of confidentiality/non-disclosure agreement), 'msa' (master "
                "services agreement), 'sow' (statement of work), 'vendor_agreement' (vendor or supplier "
                "agreement), or 'other' (the input is not a contract). "
                "Set confidence to a calibrated number in [0.0, 1.0]. "
                "Give a one-sentence reasoning. Do not call any tools and do not produce any other output."
            )}],
            "output_parser": {"type": "structured", "json_schema": classify_schema},
            "allowed_tools": [],
            "allowed_skills": [],
            "next_steps": [
                # Catch-all for now: classify always proceeds to extract.
                {"step_name": "extract"},
            ],
        },
        "extract": {
            "instructions": [{"type": "inline", "name": "extract_instructions", "template": (
                "You are a contract field extractor. The classification result from the previous step is "
                "already in this conversation's history. Read the original document and extract the requested "
                "fields. If a field is not present in the document, write '<unknown>' verbatim — do NOT invent "
                "values. Keep `notes` empty unless something is genuinely worth flagging during extraction. "
                "Do not call any tools."
            )}],
            "output_parser": {"type": "structured", "json_schema": extract_schema},
            "allowed_tools": [],
            "allowed_skills": [],
            "next_steps": [
                {"step_name": "flag_issues"},
            ],
        },
        "flag_issues": {
            "instructions": [{"type": "inline", "name": "flag_issues_instructions", "template": (
                "You are a contract risk reviewer. The classification and extracted fields are already in "
                "this conversation's history. Produce a single Markdown report with these sections:\n"
                "\n## Headline\nOne sentence: ship-as-is / negotiate-first / reject.\n"
                "\n## Risk flags\nA bulleted list. For each flag: what it is, why it matters, and how to "
                "remediate (one sentence each). If there are no material flags, write 'No material flags.'\n"
                "\n## Open questions\nFields that came back '<unknown>' that the requester should chase down. "
                "If everything was extracted, write 'None.'\n"
                "\nDo not invent terms that weren't in the source. Do not call any tools."
            )}],
            "output_parser": {"type": "default"},
            "allowed_tools": [],
            "allowed_skills": [],
            # No next_steps — flag_issues is terminal for the turn.
        },
    },
}

contract_agent_key = delete_and_create_agent(contract_agent_config, CONTRACT_AGENT_NAME)

Created agent 'Contract Triage' (key: agt_contract_triage_7a6f)


## Step 2: Run the Pipeline and Inspect the Trace

We'll send a sample NDA and watch all three phases run in a single user turn. The events worth watching:

- **`step_transition`** — `from_step` → `to_step` after a step's `next_steps` resolves. You should see two of these (`classify` → `extract`, `extract` → `flag_issues`).
- **`structured_output`** — the JSON output from a step whose `output_parser` is `structured`. We expect one each from `classify` and `extract`.
- **`agent_output`** — the free-form Markdown report from `flag_issues`.

The helper below pulls those out per cell, so you can see the pipeline shape rather than wading through the raw event list.

In [4]:
def run_pipeline(agent_key, session_key, content, show_events=True, json_preview_chars=200):
    """Send a message to an agent and pretty-print every step-transition and
    structured/agent output in the response.

    Returns the parsed events list so callers can poke at it further.
    """
    payload = {"messages": [{"type": "text", "content": content}], "stream_response": False}
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return []

    events = response.json().get("events", [])
    if show_events:
        print("\n------ Pipeline Events ------")
        for event in events:
            etype = event.get("type", "unknown")
            if etype == "step_transition":
                fr = event.get("from_step") or event.get("previous_step")
                to = event.get("to_step") or event.get("next_step") or event.get("step_name")
                print(f"[step_transition] {fr} -> {to}")
            elif etype == "structured_output":
                step = event.get("step_name") or event.get("step")
                content = event.get("content") or event.get("output") or {}
                if isinstance(content, str):
                    try:
                        content = json.loads(content)
                    except (ValueError, TypeError):
                        pass
                blob = json.dumps(content, indent=2) if isinstance(content, (dict, list)) else str(content)
                if len(blob) > json_preview_chars:
                    blob = blob[:json_preview_chars] + "..."
                tag = f" [{step}]" if step else ""
                print(f"[structured_output]{tag}\n{blob}")
            elif etype == "agent_output":
                text = (event.get("content") or "").strip()
                preview = text if len(text) <= 600 else (text[:600] + "...")
                print(f"[agent_output]\n{preview}")
            elif etype == "step_transition_limit_exceeded":
                print(f"[step_transition_limit_exceeded] hit after {event.get('count', '?')} transitions")
        print("-" * 28)
    return events

In [5]:
# Sample NDA. Deliberately short and a bit ambiguous on a couple of fields
# (no governing law clause) so the extractor has to use '<unknown>' honestly.
sample_nda = """MUTUAL NON-DISCLOSURE AGREEMENT

This Mutual Non-Disclosure Agreement ("Agreement") is entered into as of March 12, 2025
("Effective Date") between Acme Robotics, Inc., a Delaware corporation ("Acme"), and
Northwind Analytics, Inc., a California corporation ("Northwind"), each a "Party" and
collectively the "Parties".

1. Confidential Information. Each Party may disclose confidential information to the other.
2. Term. This Agreement shall remain in effect for two (2) years from the Effective Date.
3. Termination. Either Party may terminate this Agreement upon 30 days' written notice to
   the other Party. Confidentiality obligations survive for an additional 3 years after
   termination.
4. Use. Confidential information shall be used solely for the purpose of evaluating a
   potential business relationship between the Parties.

[Signature blocks omitted.]"""

# Fresh session per pipeline run.
response = requests.post(
    f"{BASE_URL}/agents/{contract_agent_key}/sessions",
    headers=headers,
    json={"name": f"Contract Triage NDA {datetime.now().strftime('%Y%m%d-%H%M%S')}",
          "metadata": {"purpose": "steps_demo_pipeline"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")
print()
print("Inbound document:")
print(sample_nda[:200] + "...")
print("=" * 80)

events = run_pipeline(contract_agent_key, session_key, sample_nda)

Session Created: ase_contract_triage_nda_20260528-152253_a434

Inbound document:
MUTUAL NON-DISCLOSURE AGREEMENT

This Mutual Non-Disclosure Agreement ("Agreement") is entered into as of March 12, 2025
("Effective Date") between Acme Robotics, Inc., a Delaware corporation ("Acme")...



------ Pipeline Events ------
[structured_output]
{
  "doc_type": "nda",
  "confidence": 1.0,
  "reasoning": "The document is titled 'Mutual Non-Disclosure Agreement' and includes terms specific to confidentiality obligations."
}
[step_transition] classify -> extract
[structured_output]
{
  "parties": [
    "Acme Robotics, Inc.",
    "Northwind Analytics, Inc."
  ],
  "effective_date": "March 12, 2025",
  "term": "2 years",
  "governing_law": "<unknown>",
  "termination_clause": "Eit...
[step_transition] extract -> flag_issues
[agent_output]
# NDA Risk Review Report

**One sentence**: Ship-as-is.

**Flags**:
- **Governing Law <unknown>**: It is important to specify the governing law to determine which jurisdiction's laws will apply in case of a dispute. Remediate by determining the agreed governing law and including it in the agreement.

**Fields that came back '<unknown>'**:
- Governing Law

No other material flags.
----------------------------


What you should see in the trace:

1. A `step_transition: classify -> extract` after the classifier emits its JSON.
2. A `structured_output` for `classify` with `doc_type: "nda"` and a high `confidence`.
3. A `step_transition: extract -> flag_issues` after extraction.
4. A `structured_output` for `extract` with `parties`, `term`, `termination_clause`, etc. — and `governing_law: "<unknown>"` because the sample NDA has no governing-law clause. (Honest is the goal; hallucinating *some* US state would have been worse.)
5. An `agent_output` from `flag_issues` — a Markdown risk report grounded only in what the prior steps emitted.

The user sent **one** message; the platform ran **three** focused LLM turns in sequence, with full session history available to each later step. That's the deterministic-pipeline shape: the system, not the model, decided which phase ran when.

## Step 3: Add a Conditional Gate

The pipeline above always runs all three phases. In production you usually want to **skip** the heavy phases when the input clearly isn't a contract — both to save tokens and to give the user a coherent answer instead of trying to extract `parties` from someone's offsite notes.

We add a fourth step, `exit_other`, and split `classify`'s `next_steps` so:

- If `doc_type == "other"` → go to `exit_other` (terminal, drafts a polite "this isn't a contract, here's what to do next" response).
- Otherwise → go to `extract` (catch-all, original behavior).

The conditional check is a UserFn expression: `get('$.output.doc_type') == 'other'`.

> ### Aside: `reentry_step`
> Each step can also set `reentry_step: "<step_name>"`. That's the step the **next** user message in the same session will enter. Without it, the session re-enters at whatever step it ended on. With it, you can build *plan once, then chat about the result*: end the analytical pipeline by setting `reentry_step` to a `qa` step, and follow-up turns hit Q&A instead of restarting from `classify`. We won't wire that here — but it's a one-line addition to `flag_issues`.

In [6]:
gated_agent_config = json.loads(json.dumps(contract_agent_config))  # deep copy
gated_agent_config["description"] = "Contract pipeline with an early-exit gate for non-contracts."

# Replace classify's next_steps with the conditional version.
gated_agent_config["steps"]["classify"]["next_steps"] = [
    {
        "condition": "get('$.output.doc_type') == 'other'",
        "step_name": "exit_other",
    },
    # Catch-all: any classification other than 'other' proceeds to extraction.
    {"step_name": "extract"},
]

# Add the exit_other step. Terminal — no next_steps.
gated_agent_config["steps"]["exit_other"] = {
    "instructions": [{"type": "inline", "name": "exit_other_instructions", "template": (
        "The classifier decided the inbound text is not a contract. "
        "Reply in 2–3 sentences explaining that this looks like something else (refer to the "
        "classifier's reasoning if helpful), and ask the user to forward an actual contract document. "
        "Do not attempt extraction or flagging. Do not call any tools."
    )}],
    "output_parser": {"type": "default"},
    "allowed_tools": [],
    "allowed_skills": [],
}

contract_agent_key = delete_and_create_agent(gated_agent_config, CONTRACT_AGENT_NAME)

# A definitely-not-a-contract input.
not_a_contract = """Hey team — quick offsite recap. We covered Q3 goals on day 1 and the new
review process on day 2. Action items: (1) Priya to circulate the calibration rubric by Friday,
(2) Sam to set up the cross-functional channel in Slack. Lunch was very good. — Lina"""

# Fresh session.
response = requests.post(
    f"{BASE_URL}/agents/{contract_agent_key}/sessions",
    headers=headers,
    json={"name": f"Contract Triage Gate {datetime.now().strftime('%Y%m%d-%H%M%S')}",
          "metadata": {"purpose": "steps_demo_gate"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")
print()
print("Inbound document (not actually a contract):")
print(not_a_contract)
print("=" * 80)

events = run_pipeline(contract_agent_key, session_key, not_a_contract)

Deleted existing agent 'Contract Triage' (agt_contract_triage_7a6f)


Created agent 'Contract Triage' (key: agt_contract_triage_f6fd)


Session Created: ase_contract_triage_gate_20260528-152306_5bc7

Inbound document (not actually a contract):
Hey team — quick offsite recap. We covered Q3 goals on day 1 and the new
review process on day 2. Action items: (1) Priya to circulate the calibration rubric by Friday,
(2) Sam to set up the cross-functional channel in Slack. Lunch was very good. — Lina



------ Pipeline Events ------
[structured_output]
{
  "doc_type": "other",
  "confidence": 0.95,
  "reasoning": "The text is an informal recap of a meeting or event, not a contract."
}
[step_transition] classify -> exit_other
[agent_output]
The document you provided appears to be a recap of an offsite meeting, which includes notes and action items rather than a contract. If you have a specific contractual document you need assistance with, please forward that, and I'll be happy to help.
----------------------------


Compare to Step 2's trace:

- `classify` runs, emits structured output with `doc_type: "other"`.
- `step_transition: classify -> exit_other` (the conditional matched first).
- `extract` and `flag_issues` **never run** — the system saved two LLM turns by routing around them.
- `exit_other` produces a polite text reply explaining what was wrong.

That's the deterministic gate: the model decides the *value* of `doc_type`; the agent config decides what happens next based on that value. The decision is in **your** system, not the LLM's narrative reasoning.

## Step 4: `reentry_step` for Follow-up Q&A

So far each user message kicks off the full pipeline (or its gated variant). For document-analysis tools you usually also want users to ask follow-up questions about what was already analyzed: *"What's the term again?"*, *"Who's the governing-law jurisdiction?"*

The lightweight way to wire that up is `reentry_step`. After `flag_issues` runs, set `reentry_step: "qa"` so the *next* user message lands in a dedicated Q&A step that has all the prior context (classification + extracted fields + flag report) but doesn't re-run the pipeline.

We'll rebuild the agent with one more step (`qa`) and a single line added to `flag_issues`.

In [7]:
reentry_agent_config = json.loads(json.dumps(gated_agent_config))
reentry_agent_config["description"] = "Contract pipeline with reentry_step Q&A for follow-up turns."

# After flag_issues finishes, the next user message in this session will land
# in `qa` instead of restarting at `classify`.
reentry_agent_config["steps"]["flag_issues"]["reentry_step"] = "qa"

# Add the qa step. No next_steps -> terminal for each follow-up turn. Re-entry
# at the same step keeps follow-ups looping back here automatically.
reentry_agent_config["steps"]["qa"] = {
    "instructions": [{"type": "inline", "name": "qa_instructions", "template": (
        "You are answering follow-up questions about a contract that was already classified, extracted, "
        "and flagged in this same session. The classification, extracted fields, and flag report are "
        "all in the conversation history. Answer the user's follow-up question succinctly using ONLY "
        "what's in history. If the requested information is genuinely not in history (e.g. a field that "
        "extraction returned as '<unknown>'), say so plainly. Do not re-run extraction. Do not invent."
    )}],
    "output_parser": {"type": "default"},
    "allowed_tools": [],
    "allowed_skills": [],
    "reentry_step": "qa",  # subsequent follow-ups also come back here
}

contract_agent_key = delete_and_create_agent(reentry_agent_config, CONTRACT_AGENT_NAME)

# Single session: turn 1 runs the full pipeline on the NDA, turn 2 is a follow-up.
response = requests.post(
    f"{BASE_URL}/agents/{contract_agent_key}/sessions",
    headers=headers,
    json={"name": f"Contract Triage Q&A {datetime.now().strftime('%Y%m%d-%H%M%S')}",
          "metadata": {"purpose": "steps_demo_reentry"}},
)
response.raise_for_status()
session_key = response.json()["key"]
print(f"Session Created: {session_key}")

# Turn 1: full pipeline.
print("\nTurn 1: analyze the NDA (full pipeline)")
print("=" * 80)
run_pipeline(contract_agent_key, session_key, sample_nda)

# Turn 2: follow-up question. Should NOT re-run classify/extract/flag_issues —
# instead it lands in `qa` because flag_issues set reentry_step="qa".
print("\nTurn 2: follow-up question (should land in qa, not classify)")
print("=" * 80)
run_pipeline(contract_agent_key, session_key, "What's the term of this NDA, and is there a survival period after termination?")

Deleted existing agent 'Contract Triage' (agt_contract_triage_f6fd)


Created agent 'Contract Triage' (key: agt_contract_triage_9594)
Session Created: ase_contract_triage_q_a_20260528-152318_f67b

Turn 1: analyze the NDA (full pipeline)



------ Pipeline Events ------
[structured_output]
{
  "doc_type": "nda",
  "confidence": 1.0,
  "reasoning": "The document is explicitly titled 'Mutual Non-Disclosure Agreement' and details confidentiality obligations."
}
[step_transition] classify -> extract
[structured_output]
{
  "parties": [
    "Acme Robotics, Inc.",
    "Northwind Analytics, Inc."
  ],
  "effective_date": "March 12, 2025",
  "term": "two (2) years from the Effective Date",
  "governing_law": "<unknown>"...
[step_transition] extract -> flag_issues
[agent_output]
# Contract Risk Review Report

**Recommendation: Negotiate-first**

- **Governing Law:**
  - **What it is:** The governing law clause specifies which jurisdiction's laws will be used to interpret the Agreement.
  - **Why it matters:** It influences the legal proceedings and is crucial in case of a dispute, affecting the cost and convenience of litigation.
  - **How to remediate:** Specify a governing law clause that is mutually agreeable to both parties.



------ Pipeline Events ------
[agent_output]
The term of the NDA is two (2) years from the Effective Date. Yes, there is a survival period after termination; confidentiality obligations survive for an additional 3 years after termination.
----------------------------


[{'id': 'aev_ba385985-372a-4eb1-a534-ee8c3993cc14',
  'session_key': 'ase_contract_triage_q_a_20260528-152318_f67b',
  'created_at': '2026-05-28T22:23:26.445Z',
  'type': 'input_message',
  'messages': [{'type': 'text',
    'content': "What's the term of this NDA, and is there a survival period after termination?"}],
  'message_diffs': ['']},
 {'id': 'aev_70e7de9e-2b0a-4fd1-a4a5-f928c698752e',
  'session_key': 'ase_contract_triage_q_a_20260528-152318_f67b',
  'created_at': '2026-05-28T22:23:27.094Z',
  'type': 'agent_output',
  'content': 'The term of the NDA is two (2) years from the Effective Date. Yes, there is a survival period after termination; confidentiality obligations survive for an additional 3 years after termination.'}]

The trace shows the difference clearly:

- **Turn 1** runs the whole pipeline: two `step_transition` lines, two `structured_output` lines, one `agent_output`.
- **Turn 2** has **zero** `step_transition` lines — Vectara entered directly at `qa` because that's what the prior step's `reentry_step` named. The agent answers the follow-up using fields already in session history, without re-extracting anything.

Two ways `reentry_step` keeps things deterministic:
- **Cost.** Pipelines aren't free; you don't want every "what was that date again?" follow-up to re-run classification + extraction.
- **Consistency.** The Q&A reply is grounded in the same extraction the user already saw. Re-running extraction would risk producing a slightly different `<unknown>` set and confusing the user.

## Step 5: When to Reach for Steps (Decision Aid)

- **Steps** — a workflow has genuinely distinct phases (classify, extract, flag) where each phase benefits from its own focused prompt and tool set, *and* you want the system (not the LLM) deciding which phase runs when. Today's notebook.
- **Sub-agents** — the work needs its own *isolated* context (start fresh), its own model, or a heavyweight tool surface. Sub-agents do **not** share session history with the parent. See notebook 6.
- **Skills** (notebook 13) — the agent should sometimes adopt a different *mindset* for the same job. Skills load instructions on demand within a single conversation; they don't change which phase runs.
- **One big prompt** — the workflow really is one phase. If you're not branching on intermediate outputs and not changing tool access, you don't need steps.

### Anti-patterns

- **Over-stepping.** Splitting trivial transitions ("first say hello, then answer") into separate steps adds latency and cost without buying determinism. Fewer, denser steps win.
- **Conditions on text fields.** `get('$.output.text') == 'sales'` is fragile — text matches break on whitespace, casing, model phrasing. Use `output_parser: "structured"` and route on a typed enum field.
- **Routing logic in the LLM prompt.** *"…and if the doc looks like an NDA, then go to the next step…"* puts deterministic flow back into a non-deterministic place. Keep flow in `next_steps`; keep the model focused on the work inside its step.
- **Forgotten reentry.** If a session is meant to support follow-ups, set `reentry_step` explicitly. Without it, the session re-enters wherever it ended — usually fine, but easy to be surprised by.

## Cleanup (Optional)

Delete the agent created in this notebook so you don't accumulate test resources.

In [8]:
if contract_agent_key:
    response = requests.delete(f"{BASE_URL}/agents/{contract_agent_key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted Contract Triage agent: {contract_agent_key}")
    else:
        print(f"Error deleting {contract_agent_key}: {response.status_code} - {response.text}")

Deleted Contract Triage agent: agt_contract_triage_9594
